<a href="https://colab.research.google.com/github/maddi-venkata-teja/Algoverse-Chatbot/blob/main/A24126552142_tanusree.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
from typing import List, Tuple

print("✅ Imports done. No pip install needed!")

# ==================== CONSTANTS ====================

# Regex pattern — single-line comments  (#  or  //)
PATTERN_SINGLE_COMMENT  = r"(#|//).*"

# Regex pattern — multi-line / block comments  (/* ... */)
PATTERN_BLOCK_COMMENT   = r"/\*.*?\*/"

# Regex pattern — docstrings  (\"\"\"...\"\"\"  or  \'\'\'...\'\'\'  )
PATTERN_DOCSTRING        = r'(""".*?"""|\'\'\'.*?\'\'\')'

# Regex pattern — one or more blank lines  →  collapse to single blank line
PATTERN_BLANK_LINES      = r"\n\s*\n"

# Regex pattern — trailing whitespace on every line
PATTERN_TRAILING_WS      = r"[ \t]+$"

# Regex pattern — Python / JS variable & function identifiers
PATTERN_IDENTIFIER       = r"\b[a-zA-Z_][a-zA-Z0-9_]*\b"

# Regex pattern — hard-coded secrets (api_key / password / token assignments)
PATTERN_SECRET           = r'(?i)(api_key|password|token|secret)\s*=\s*["\'].*?["\']'

# Regex pattern — import statements (Python)
PATTERN_IMPORT           = r"^\s*(import|from)\s+\S+"

# Regex pattern — URLs  (http / https)
PATTERN_URL              = r"https?://[^\s\'\"]+"

print("✅ Constants defined.")

# ============================================================
# Cell 2 — Sample DSA Code (used as input for all demos)
# ============================================================

SAMPLE_CODE = '''\
# ---- Binary Search implementation ----
def binary_search(arr, target):
    """
    Returns the index of target in sorted array arr.
    Returns -1 if not found.
    """
    api_key = "sk-ABC123SECRET"          # TODO: move to env
    password = 'hunter2'
    left, right = 0, len(arr) - 1       // initialise pointers

    while left <= right:
        mid = (left + right) // 2       /* integer midpoint */
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            left  = mid + 1
        else:
            right = mid - 1

    return -1    # target not present


# ---- Quick Sort implementation ----
def quick_sort(arr):
    \'\'\'In-place quicksort using last element as pivot.\'\'\'
    import random           # stdlib — used for shuffle
    from os import path     # unused import left in by mistake

    if len(arr) <= 1:
        return arr
    pivot   = arr[-1]
    lesser  = [x for x in arr[:-1] if x <= pivot]
    greater = [x for x in arr[:-1] if x >  pivot]
    return quick_sort(lesser) + [pivot] + quick_sort(greater)


# Reference: https://en.wikipedia.org/wiki/Quicksort
'''

print("✅ Sample DSA code loaded.")
print(f"   Total characters : {len(SAMPLE_CODE)}")
print(f"   Total lines      : {len(SAMPLE_CODE.splitlines())}")

# ============================================================
# Cell 3 — Function Definitions
# ============================================================

# ==================== 1. REMOVE SINGLE-LINE COMMENTS ====================

def remove_single_line_comments(code: str) -> str:
    """
    Strips all single-line comments from source code.

    Matches everything from  #  or  //  to the end of the line.
    Works for Python (#) and C-style (//) languages.

    re.MULTILINE makes ^ / $ match per line instead of whole string.
    """
    cleaned = re.sub(PATTERN_SINGLE_COMMENT, "", code, flags=re.MULTILINE)
    print("[remove_single_line_comments] ✅ Single-line comments removed.")
    return cleaned


# ==================== 2. REMOVE BLOCK COMMENTS ====================

def remove_block_comments(code: str) -> str:
    """
    Strips C-style block comments  /* ... */  from source code.

    re.DOTALL makes  .  match newline characters too, so multi-line
    block comments are fully consumed in one pass.
    """
    cleaned = re.sub(PATTERN_BLOCK_COMMENT, "", code, flags=re.DOTALL)
    print("[remove_block_comments] ✅ Block comments removed.")
    return cleaned


# ==================== 3. REMOVE DOCSTRINGS ====================

def remove_docstrings(code: str) -> str:
    """
    Strips Python triple-quoted docstrings  \"\"\"...\"\"\"  and  \'\'\'...\'\'\'

    re.DOTALL is required so the pattern crosses line boundaries.
    Both double-quote and single-quote variants are handled by the
    alternation inside PATTERN_DOCSTRING.
    """
    cleaned = re.sub(PATTERN_DOCSTRING, "", code, flags=re.DOTALL)
    print("[remove_docstrings] ✅ Docstrings removed.")
    return cleaned


# ==================== 4. NORMALIZE WHITESPACE ====================

def normalize_whitespace(code: str) -> str:
    """
    Two-pass whitespace cleanup:

    Pass 1 — strip trailing spaces / tabs from every line.
             re.MULTILINE makes $ match at each line-end.

    Pass 2 — collapse runs of 2+ blank lines into a single blank line.
             re.DOTALL lets  .  span newlines; \\n\\n replaces the gap.
    """
    cleaned = re.sub(PATTERN_TRAILING_WS, "", code, flags=re.MULTILINE)
    cleaned = re.sub(PATTERN_BLANK_LINES, "\n\n", cleaned)
    print("[normalize_whitespace] ✅ Whitespace normalized.")
    return cleaned


# ==================== 5. EXTRACT IDENTIFIERS ====================

def extract_identifiers(code: str) -> List[str]:
    """
    Finds all Python / JS-style identifiers in the code.

    PATTERN_IDENTIFIER matches tokens that start with a letter or
    underscore, followed by any mix of letters, digits, underscores.

    re.findall returns every non-overlapping match as a plain string.
    Duplicates are removed with set(); result is sorted alphabetically.
    """
    identifiers = re.findall(PATTERN_IDENTIFIER, code)
    unique_ids  = sorted(set(identifiers))
    print(f"[extract_identifiers] ✅ Found {len(unique_ids)} unique identifier(s).")
    return unique_ids


# ==================== 6. REDACT SECRETS ====================

def redact_secrets(code: str) -> str:
    """
    Replaces hard-coded credentials with a safe placeholder.

    Matches  api_key = "..."  /  password = '...'  /  token = "..."
    (case-insensitive via re.IGNORECASE) and substitutes the value
    with  "***REDACTED***"  so logs and embeddings stay clean.
    """
    redacted = re.sub(
        PATTERN_SECRET,
        lambda m: f'{m.group(1)} = "***REDACTED***"',
        code,
        flags=re.IGNORECASE,
    )
    print("[redact_secrets] ✅ Secrets redacted.")
    return redacted


# ==================== 7. EXTRACT IMPORTS ====================

def extract_imports(code: str) -> List[str]:
    """
    Pulls out every  import  /  from ... import  statement.

    re.MULTILINE is needed so  ^  anchors to the start of each line,
    not just the start of the whole string.

    Returns a deduplicated, sorted list of import keywords found.
    """
    imports = re.findall(PATTERN_IMPORT, code, flags=re.MULTILINE)
    print(f"[extract_imports] ✅ Found {len(imports)} import statement(s).")
    return imports


# ==================== 8. EXTRACT URLS ====================

def extract_urls(code: str) -> List[str]:
    """
    Scans code (including comments) for http / https URLs.

    Stops the match at whitespace or quote characters so it doesn't
    accidentally swallow surrounding syntax.

    Returns a list of all URLs found (may contain duplicates).
    """
    urls = re.findall(PATTERN_URL, code)
    print(f"[extract_urls] ✅ Found {len(urls)} URL(s).")
    return urls


# ==================== 9. FULL PIPELINE ====================

def clean_code(code: str) -> Tuple[str, dict]:
    """
    Runs the complete code-cleaning pipeline in order:

      1. Redact secrets       — before anything else to avoid leaks
      2. Remove docstrings    — triple-quoted blocks
      3. Remove block comments — /* ... */
      4. Remove single-line comments — # and //
      5. Normalize whitespace — trailing spaces + blank lines

    Also extracts metadata (identifiers, imports, URLs) from the
    original code for reporting / downstream use.

    Returns
    -------
    cleaned  : str   — sanitized code string
    metadata : dict  — extracted identifiers, imports, URLs
    """
    print("\n[clean_code] 🚀 Starting full cleaning pipeline...")

    # -- metadata extraction (run on raw code BEFORE stripping) --
    identifiers = extract_identifiers(code)
    imports     = extract_imports(code)
    urls        = extract_urls(code)

    # -- cleaning passes --
    cleaned = redact_secrets(code)
    cleaned = remove_docstrings(cleaned)
    cleaned = remove_block_comments(cleaned)
    cleaned = remove_single_line_comments(cleaned)
    cleaned = normalize_whitespace(cleaned)

    metadata = {
        "identifiers" : identifiers,
        "imports"     : imports,
        "urls"        : urls,
    }

    print("[clean_code] ✅ Pipeline complete.")
    return cleaned, metadata


# ==================== 10. STATS ====================

def print_cleaning_stats(original: str, cleaned: str, metadata: dict) -> None:
    """Prints a before/after summary of the cleaning run."""
    original_lines = original.strip().splitlines()
    cleaned_lines  = cleaned.strip().splitlines()

    print("\n" + "=" * 55)
    print("  REGEX CODE CLEANING STATS")
    print("=" * 55)
    print(f"  Original chars          : {len(original)}")
    print(f"  Cleaned  chars          : {len(cleaned)}")
    print(f"  Chars removed           : {len(original) - len(cleaned)}")
    print(f"  Original lines          : {len(original_lines)}")
    print(f"  Cleaned  lines          : {len(cleaned_lines)}")
    print(f"  Lines removed           : {len(original_lines) - len(cleaned_lines)}")
    print(f"  Unique identifiers      : {len(metadata['identifiers'])}")
    print(f"  Import statements       : {len(metadata['imports'])}")
    print(f"  URLs found              : {len(metadata['urls'])}")
    print("=" * 55)

    if metadata["urls"]:
        print("\n  URLs extracted:")
        for url in metadata["urls"]:
            print(f"    - {url}")

    if metadata["imports"]:
        print("\n  Import keywords found:")
        for kw in metadata["imports"]:
            print(f"    - {kw}")

    print("\n  Sample identifiers (first 10):")
    for ident in metadata["identifiers"][:10]:
        print(f"    - {ident}")


print("✅ All functions defined successfully!")

# ============================================================
# Cell 4 — Run Step by Step (Individual Demos)
# ============================================================

print("=" * 55)
print("  REGEX FOR CODE CLEANING  —  [By I.Tanusree]  [A24126552142]")
print("  Week 2: regex for code cleaning")
print("=" * 55)

# Step 1 — Redact secrets
print("\n--- Step 1: Redact Secrets ---")
redacted = redact_secrets(SAMPLE_CODE)
print("\nBEFORE:")
print('  api_key = "sk-ABC123SECRET"')
print("AFTER:")
for line in redacted.splitlines():
    if "REDACTED" in line:
        print(f"  {line.strip()}")

# Step 2 — Remove docstrings
print("\n--- Step 2: Remove Docstrings ---")
no_docs = remove_docstrings(SAMPLE_CODE)

# Step 3 — Remove block comments
print("\n--- Step 3: Remove Block Comments ---")
no_blocks = remove_block_comments(SAMPLE_CODE)

# Step 4 — Remove single-line comments
print("\n--- Step 4: Remove Single-Line Comments ---")
no_single = remove_single_line_comments(SAMPLE_CODE)

# Step 5 — Normalize whitespace
print("\n--- Step 5: Normalize Whitespace ---")
normalized = normalize_whitespace(SAMPLE_CODE)

# Step 6 — Extract identifiers
print("\n--- Step 6: Extract Identifiers ---")
identifiers = extract_identifiers(SAMPLE_CODE)
print(f"  First 10: {identifiers[:10]}")

# Step 7 — Extract imports
print("\n--- Step 7: Extract Imports ---")
imports = extract_imports(SAMPLE_CODE)
print(f"  Found: {imports}")

# Step 8 — Extract URLs
print("\n--- Step 8: Extract URLs ---")
urls = extract_urls(SAMPLE_CODE)
print(f"  Found: {urls}")

# ============================================================
# Cell 5 — Full Pipeline + Stats + Final Output
# ============================================================

# Step 9 — Full pipeline
print("\n--- Step 9: Full Cleaning Pipeline ---")
cleaned, metadata = clean_code(SAMPLE_CODE)

# Step 10 — Stats
print("\n--- Step 10: Stats ---")
print_cleaning_stats(SAMPLE_CODE, cleaned, metadata)

# Step 11 — Final cleaned output
print("\n--- Step 11: Cleaned Code Output ---")
print("-" * 55)
print(cleaned.strip())
print("-" * 55)
print("\n✅ Week 2 — Regex for Code Cleaning complete!")


✅ Imports done. No pip install needed!
✅ Constants defined.
✅ Sample DSA code loaded.
   Total characters : 1130
   Total lines      : 37
✅ All functions defined successfully!
  REGEX FOR CODE CLEANING  —  [Your Name]  [Roll No]
  Week 2: Text Processing

--- Step 1: Redact Secrets ---
[redact_secrets] ✅ Secrets redacted.

BEFORE:
  api_key = "sk-ABC123SECRET"
AFTER:
  api_key = "***REDACTED***"          # TODO: move to env
  password = "***REDACTED***"

--- Step 2: Remove Docstrings ---
[remove_docstrings] ✅ Docstrings removed.

--- Step 3: Remove Block Comments ---
[remove_block_comments] ✅ Block comments removed.

--- Step 4: Remove Single-Line Comments ---
[remove_single_line_comments] ✅ Single-line comments removed.

--- Step 5: Normalize Whitespace ---
[normalize_whitespace] ✅ Whitespace normalized.

--- Step 6: Extract Identifiers ---
[extract_identifiers] ✅ Found 72 unique identifier(s).
  First 10: ['ABC123SECRET', 'Binary', 'In', 'Quick', 'Quicksort', 'Reference', 'Returns', 